In [1]:
import sys
sys.path.append('..')
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from src.decision_tree import DecisionTree
from src.random_forest import RandomForest

In [2]:
# Load data
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]}")
print(f"Features: {X_train.shape[1]}")

Training samples: 16512
Features: 8


In [9]:
# Train single decision tree
tree = DecisionTree(mode='regression', max_depth=10)
tree.fit(X_train, y_train)
tree_predictions = tree.predict(X_test)

In [8]:
# Train random forest
forest = RandomForest(mode='regression', tree_count=20)
forest.fit(X_train, y_train)
forest_predictions = forest.predict(X_test)

KeyboardInterrupt: 

In [4]:
# Evaluate both
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

def r_squared(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    ss_residual = np.sum((y_true - y_pred)**2)
    ss_total = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (ss_residual / ss_total)


In [5]:

print("Decision Tree:")
print(f"  RMSE: {round(rmse(y_test, tree_predictions), 4)}")
print(f"  R²:   {round(r_squared(y_test, tree_predictions), 4)}")

print("Random Forest:")
print(f"  RMSE: {round(rmse(y_test, forest_predictions), 4)}")
print(f"  R²:   {round(r_squared(y_test, forest_predictions), 4)}")

Decision Tree:
  RMSE: 0.7242
  R²:   0.5998
Random Forest:
  RMSE: 0.5313
  R²:   0.7846


In [11]:
K = 5
folds_X = np.array_split(X_train, K)
folds_y = np.array_split(y_train, K)

scores = []
for i in range(K):
    X_test_fold  = folds_X[i]
    y_test_fold  = folds_y[i]
    X_train_fold = np.concatenate([folds_X[j] for j in range(K) if j != i])
    y_train_fold = np.concatenate([folds_y[j] for j in range(K) if j != i])
    forest = RandomForest(mode="regression", tree_count=20)
    forest.fit(X_train_fold, y_train_fold)
    forest_predictions = forest.predict(X_test_fold)
    forest_rmse = round(rmse(y_test_fold, forest_predictions), 4)
    scores.append(forest_rmse)
print(f"Average RMSE across {K} folds: {round(np.mean(scores), 4)}")
print(f"Scores per fold: {scores}")

Average RMSE across 5 folds: 0.5349
Scores per fold: [np.float64(0.5415), np.float64(0.5343), np.float64(0.5325), np.float64(0.5294), np.float64(0.5368)]
